# peS2o Deduplication Training-Efficiency Curves

Run this notebook after the `raw`, `minhashlsh`, and `lshbloom` full-corpus training runs are complete. It downloads the three measured result files from S3, validates checkpoint ordering and experiment identity, creates CSV files and four comparison plots, and uploads the outputs to W&B and S3.

This notebook does not require a GPU. Add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `WANDB_API_KEY` to Colab Secrets. Temporary AWS credentials also require `AWS_SESSION_TOKEN`.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "boto3>=1.35,<2",
    "wandb>=0.21,<1",
    "pandas>=2.2,<3",
    "matplotlib>=3.9,<4",
])
print("Dependencies installed.")


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
from collections.abc import Iterable, Mapping
from pathlib import Path

VARIANTS = ("raw", "minhashlsh", "lshbloom")
CURVE_COLUMNS = (
    "variant",
    "global_step",
    "cumulative_train_tokens",
    "cumulative_train_gpu_hours",
    "validation_loss",
    "validation_perplexity",
    "sciq_acc",
    "sciq_acc_norm",
    "sciq_acc_stderr",
    "sciq_acc_norm_stderr",
    "sciq_examples",
)


def canonical_sha256(value: object) -> str:
    """Return a stable SHA-256 for JSON-compatible experiment metadata."""
    try:
        encoded = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    except (TypeError, ValueError) as error:
        raise ValueError(
            "value must contain only finite JSON-compatible data"
        ) from error
    return hashlib.sha256(encoded).hexdigest()


def validate_shared_experiment_results(
    results: Iterable[Mapping[str, object]],
    expected_variants: Iterable[str] = VARIANTS,
) -> str:
    """Verify that separately produced variant results belong to one experiment."""
    results = [dict(result) for result in results]
    expected_variants = tuple(expected_variants)
    if len(results) != len(expected_variants):
        raise ValueError("one experiment result is required for every variant")

    variants = [result.get("variant") for result in results]
    if set(variants) != set(expected_variants) or len(set(variants)) != len(variants):
        raise ValueError(
            "experiment result variants do not match the expected variants"
        )

    verified = []
    for result in results:
        if result.get("schema_version") != 2:
            raise ValueError("experiment results require schema_version 2")
        identity = result.get("experiment_identity")
        fingerprint = result.get("experiment_fingerprint")
        if not isinstance(identity, Mapping) or not isinstance(fingerprint, str):
            raise ValueError("experiment identity and fingerprint are required")
        computed = canonical_sha256(identity)
        if computed != fingerprint:
            raise ValueError(
                f"experiment fingerprint is invalid for variant {result.get('variant')}"
            )
        verified.append(fingerprint)

    if len(set(verified)) != 1:
        raise ValueError("experiment fingerprints differ across variants")
    return verified[0]


def _positive_integer(value: object, name: str) -> int:
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive integer")
    try:
        converted = int(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a positive integer") from error
    if converted < 1 or converted != value:
        raise ValueError(f"{name} must be a positive integer")
    return converted


def _finite_number(value: object, name: str) -> float:
    try:
        converted = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a finite number") from error
    if not math.isfinite(converted):
        raise ValueError(f"{name} must be a finite number")
    return converted


def build_full_training_plan(
    token_counts: Mapping[str, int], sequence_length: int
) -> dict[str, dict[str, int]]:
    sequence_length = _positive_integer(sequence_length, "sequence_length")
    if not token_counts:
        raise ValueError("token_counts must not be empty")

    plan = {}
    for variant, raw_count in token_counts.items():
        available_tokens = _positive_integer(
            raw_count, f"available tokens for {variant}"
        )
        sequence_count = available_tokens // sequence_length
        if sequence_count < 1:
            raise ValueError(
                f"{variant} has no complete sequence of length {sequence_length}"
            )
        train_input_tokens = sequence_count * sequence_length
        plan[str(variant)] = {
            "available_tokens": available_tokens,
            "sequence_count": sequence_count,
            "train_input_tokens": train_input_tokens,
            "unused_tail_tokens": available_tokens - train_input_tokens,
        }
    return plan


def checkpoint_steps(total_optimizer_steps: int, save_steps: int) -> list[int]:
    total_optimizer_steps = _positive_integer(
        total_optimizer_steps, "total_optimizer_steps"
    )
    save_steps = _positive_integer(save_steps, "save_steps")
    steps = [0]
    steps.extend(range(save_steps, total_optimizer_steps + 1, save_steps))
    if steps[-1] != total_optimizer_steps:
        steps.append(total_optimizer_steps)
    return steps


def tokens_at_step(
    global_step: int,
    tokens_per_optimizer_step: int,
    total_train_tokens: int,
) -> int:
    if isinstance(global_step, bool):
        raise ValueError("global_step must be a non-negative integer")
    try:
        global_step = int(global_step)
    except (TypeError, ValueError) as error:
        raise ValueError("global_step must be a non-negative integer") from error
    if global_step < 0:
        raise ValueError("global_step must be a non-negative integer")
    tokens_per_optimizer_step = _positive_integer(
        tokens_per_optimizer_step, "tokens_per_optimizer_step"
    )
    total_train_tokens = _positive_integer(total_train_tokens, "total_train_tokens")
    return min(global_step * tokens_per_optimizer_step, total_train_tokens)


def _index_measurements(
    rows: Iterable[dict], label: str
) -> dict[tuple[str, int], dict]:
    indexed = {}
    for row in rows:
        if not isinstance(row, dict):
            raise ValueError(f"each {label} measurement must be an object")
        variant = row.get("variant")
        if not isinstance(variant, str) or not variant:
            raise ValueError(f"each {label} measurement requires variant")
        step = row.get("global_step")
        if isinstance(step, bool):
            raise ValueError(f"each {label} measurement requires integer global_step")
        try:
            step = int(step)
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"each {label} measurement requires integer global_step"
            ) from error
        if step < 0:
            raise ValueError(
                f"each {label} measurement requires non-negative global_step"
            )
        key = (variant, step)
        if key in indexed:
            raise ValueError(f"duplicate {label} measurement: {variant} step {step}")
        indexed[key] = dict(row, global_step=step)
    return indexed


def merge_curve_measurements(
    validation_rows: Iterable[dict], sciq_rows: Iterable[dict]
) -> list[dict]:
    validation = _index_measurements(validation_rows, "validation")
    sciq = _index_measurements(sciq_rows, "SciQ")
    if validation.keys() != sciq.keys():
        missing_sciq = sorted(validation.keys() - sciq.keys())
        missing_validation = sorted(sciq.keys() - validation.keys())
        raise ValueError(
            "validation and SciQ measurement keys differ: "
            f"missing SciQ={missing_sciq}, missing validation={missing_validation}"
        )

    merged = []
    for key in sorted(
        validation,
        key=lambda item: (
            VARIANTS.index(item[0]) if item[0] in VARIANTS else len(VARIANTS),
            item[0],
            item[1],
        ),
    ):
        row = dict(validation[key])
        for field, value in sciq[key].items():
            if field in ("variant", "global_step"):
                continue
            if field in row and row[field] != value:
                raise ValueError(f"conflicting field {field!r} for {key}")
            row[field] = value
        merged.append(row)

    expected_variants = tuple(dict.fromkeys(row["variant"] for row in merged))
    validate_curve_rows(merged, expected_variants=expected_variants)
    return merged


def validate_curve_rows(
    rows: Iterable[dict], expected_variants: Iterable[str] = VARIANTS
) -> list[dict]:
    rows = [dict(row) for row in rows]
    expected_variants = tuple(expected_variants)
    if not rows:
        raise ValueError("curve rows must not be empty")

    by_variant = {variant: [] for variant in expected_variants}
    for row in rows:
        missing = [column for column in CURVE_COLUMNS if column not in row]
        if missing:
            raise ValueError(f"curve row is missing columns: {', '.join(missing)}")
        variant = row["variant"]
        if variant not in by_variant:
            raise ValueError(f"unexpected variant: {variant}")
        by_variant[variant].append(row)

        for field in (
            "global_step",
            "cumulative_train_tokens",
            "sciq_examples",
        ):
            value = _finite_number(row[field], field)
            if value < 0 or int(value) != value:
                raise ValueError(f"{field} must be a non-negative integer")
        for field in (
            "cumulative_train_gpu_hours",
            "validation_loss",
            "validation_perplexity",
            "sciq_acc",
            "sciq_acc_norm",
            "sciq_acc_stderr",
            "sciq_acc_norm_stderr",
        ):
            value = _finite_number(row[field], field)
            if value < 0:
                raise ValueError(f"{field} must be non-negative")
        if not 0 <= float(row["sciq_acc"]) <= 1:
            raise ValueError("sciq_acc must be between zero and one")
        if not 0 <= float(row["sciq_acc_norm"]) <= 1:
            raise ValueError("sciq_acc_norm must be between zero and one")

    for variant, variant_rows in by_variant.items():
        if not variant_rows:
            raise ValueError(f"missing variant: {variant}")
        variant_rows.sort(key=lambda row: int(row["global_step"]))
        if int(variant_rows[0]["global_step"]) != 0:
            raise ValueError(f"{variant} curve must begin at global step zero")
        for previous, current in zip(variant_rows, variant_rows[1:]):
            if int(current["global_step"]) <= int(previous["global_step"]):
                raise ValueError(f"{variant} global steps must strictly increase")
            if int(current["cumulative_train_tokens"]) <= int(
                previous["cumulative_train_tokens"]
            ):
                raise ValueError(f"{variant} token counts must strictly increase")
            if float(current["cumulative_train_gpu_hours"]) <= float(
                previous["cumulative_train_gpu_hours"]
            ):
                raise ValueError(f"{variant} GPU hours must strictly increase")
    return rows


def write_curve_csv(path: str | Path, rows: Iterable[dict]) -> Path:
    rows = [dict(row) for row in rows]
    if not rows:
        raise ValueError("curve rows must not be empty")
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=CURVE_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    return output_path


## 1. Download and validate the three real experiment results

In [ ]:
import json
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import wandb
from google.colab import userdata


CONFIG = {
    "wandb_project": "lshbloom-pes2o",
    "wandb_group": "qwen2.5-0.5b-pes2o-dedup-full-efficiency",
    "wandb_run_name": "qwen2.5-0.5b-full-efficiency-curves",
}
S3_BUCKET = "calista-bucket"
S3_PREFIX = "pes2o/v2/experiments/pilot-5000/efficiency/"
RESULT_KEYS = {
    "raw": f"{S3_PREFIX}raw/curve-results.json",
    "minhashlsh": f"{S3_PREFIX}minhashlsh/curve-results.json",
    "lshbloom": f"{S3_PREFIX}lshbloom/curve-results.json",
}
OUTPUT_DIR = Path("/content/results/efficiency-curves")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def optional_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
for name, value in (
    ("AWS_ACCESS_KEY_ID", AWS_ACCESS_KEY_ID),
    ("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY),
    ("WANDB_API_KEY", WANDB_API_KEY),
):
    if not value:
        raise RuntimeError(f"Missing required Colab secret: {name}")

session_kwargs = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region_name": optional_secret("AWS_DEFAULT_REGION") or "ap-northeast-1",
}
aws_session_token = optional_secret("AWS_SESSION_TOKEN")
if aws_session_token:
    session_kwargs["aws_session_token"] = aws_session_token
s3 = boto3.session.Session(**session_kwargs).client("s3")

all_rows = []
raw_results = {}
for variant in VARIANTS:
    destination = OUTPUT_DIR / f"{variant}-curve-results.json"
    s3.download_file(S3_BUCKET, RESULT_KEYS[variant], str(destination))
    result = json.loads(destination.read_text(encoding="utf-8"))
    if result.get("variant") != variant:
        raise RuntimeError(
            f"Expected {variant} result, received {result.get('variant')}"
        )
    rows = result["curve_probe"]["rows"]
    validate_curve_rows(rows, expected_variants=(variant,))
    all_rows.extend(rows)
    raw_results[variant] = result

experiment_fingerprint = validate_shared_experiment_results(raw_results.values())
validate_curve_rows(all_rows, expected_variants=VARIANTS)
frame = pd.DataFrame(all_rows, columns=CURVE_COLUMNS)
frame.to_csv(OUTPUT_DIR / "all-curve-results.csv", index=False)
print("Verified shared experiment fingerprint:", experiment_fingerprint)
display(frame)


## 2. Calculate final cost and quality differences

Token savings use Raw as the baseline. GPU-hour savings use measured training time and can differ slightly from token savings because throughput is not perfectly constant.


In [ ]:
final_rows = (
    frame.sort_values(["variant", "global_step"])
    .groupby("variant", sort=False)
    .tail(1)
    .set_index("variant")
    .loc[list(VARIANTS)]
    .reset_index()
)
raw_tokens = float(final_rows.loc[final_rows["variant"] == "raw", "cumulative_train_tokens"].iloc[0])
raw_hours = float(final_rows.loc[final_rows["variant"] == "raw", "cumulative_train_gpu_hours"].iloc[0])
raw_acc = float(final_rows.loc[final_rows["variant"] == "raw", "sciq_acc_norm"].iloc[0])
raw_ppl = float(final_rows.loc[final_rows["variant"] == "raw", "validation_perplexity"].iloc[0])

summary = final_rows[[
    "variant",
    "cumulative_train_tokens",
    "cumulative_train_gpu_hours",
    "validation_perplexity",
    "sciq_acc_norm",
]].copy()
summary["token_saving_vs_raw"] = 1.0 - summary["cumulative_train_tokens"] / raw_tokens
summary["gpu_hour_saving_vs_raw"] = 1.0 - summary["cumulative_train_gpu_hours"] / raw_hours
summary["delta_sciq_acc_norm_vs_raw"] = summary["sciq_acc_norm"] - raw_acc
summary["delta_validation_perplexity_vs_raw"] = summary["validation_perplexity"] - raw_ppl
summary.to_csv(OUTPUT_DIR / "final-efficiency-summary.csv", index=False)
display(summary)


## 3. Plot perplexity and SciQ accuracy against tokens and GPU hours

In [ ]:
LABELS = {
    "raw": "Raw",
    "minhashlsh": "MinHashLSH",
    "lshbloom": "LSHBloom",
}
COLORS = {
    "raw": "#4C78A8",
    "minhashlsh": "#F58518",
    "lshbloom": "#54A24B",
}

plt.style.use("seaborn-v0_8-whitegrid")
figure, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
plot_specs = (
    ("cumulative_train_tokens", "validation_perplexity", "Cumulative training tokens (millions)", "Validation perplexity", 1e6),
    ("cumulative_train_tokens", "sciq_acc_norm", "Cumulative training tokens (millions)", "SciQ normalized accuracy (%)", 1e6),
    ("cumulative_train_gpu_hours", "validation_perplexity", "Cumulative training GPU hours", "Validation perplexity", 1.0),
    ("cumulative_train_gpu_hours", "sciq_acc_norm", "Cumulative training GPU hours", "SciQ normalized accuracy (%)", 1.0),
)

for axis, (x_column, y_column, x_label, y_label, x_divisor) in zip(axes.flat, plot_specs):
    for variant in VARIANTS:
        subset = frame[frame["variant"] == variant].sort_values("global_step")
        x_values = subset[x_column] / x_divisor
        if y_column == "sciq_acc_norm":
            y_values = subset[y_column] * 100.0
            y_errors = subset["sciq_acc_norm_stderr"] * 100.0
            axis.errorbar(
                x_values,
                y_values,
                yerr=y_errors,
                color=COLORS[variant],
                marker="o",
                markersize=4,
                linewidth=2,
                capsize=2,
                label=LABELS[variant],
            )
        else:
            y_values = subset[y_column]
            axis.plot(
                x_values,
                y_values,
                color=COLORS[variant],
                marker="o",
                markersize=4,
                linewidth=2,
                label=LABELS[variant],
            )
        axis.scatter(
            [x_values.iloc[-1]],
            [y_values.iloc[-1]],
            color=COLORS[variant],
            marker="D",
            s=42,
            zorder=4,
        )
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.set_title(f"{y_label} vs. {x_label.lower()}")
    axis.grid(True, alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
figure.suptitle(
    "Qwen2.5-0.5B continued pretraining on complete peS2o variants",
    fontsize=15,
    y=1.03,
)
figure.text(
    0.5,
    -0.02,
    "Diamonds mark the end of one epoch. Accuracy bars are lm-eval standard errors.",
    ha="center",
    fontsize=10,
)

png_path = OUTPUT_DIR / "efficiency-curves.png"
pdf_path = OUTPUT_DIR / "efficiency-curves.pdf"
figure.savefig(png_path, dpi=200, bbox_inches="tight")
figure.savefig(pdf_path, bbox_inches="tight")
plt.show()


## 4. Upload the comparison to W&B and S3

In [ ]:
wandb.login(key=WANDB_API_KEY)
run = wandb.init(
    project=CONFIG["wandb_project"],
    group=CONFIG["wandb_group"],
    name=CONFIG["wandb_run_name"],
    config={
        **CONFIG,
        "source_results": RESULT_KEYS,
        "experiment_fingerprint": experiment_fingerprint,
    },
)
run.log({
    "efficiency/curves": wandb.Image(str(png_path)),
    "efficiency/all_points": wandb.Table(dataframe=frame),
    "efficiency/final_summary": wandb.Table(dataframe=summary),
})

artifact = wandb.Artifact(
    name="qwen2.5-0.5b-pes2o-full-efficiency-curves",
    type="evaluation",
)
artifact.add_dir(str(OUTPUT_DIR))
run.log_artifact(artifact)

summary_prefix = f"{S3_PREFIX}summary/"
for local_path in OUTPUT_DIR.iterdir():
    if local_path.is_file():
        s3.upload_file(str(local_path), S3_BUCKET, f"{summary_prefix}{local_path.name}")

run.summary["results_s3_uri"] = f"s3://{S3_BUCKET}/{summary_prefix}"
run.summary["experiment_fingerprint"] = experiment_fingerprint
print("W&B:", run.url)
print("S3:", run.summary["results_s3_uri"])
wandb.finish()


## Reading the curves

A useful efficiency result has two properties: the deduplicated curve reaches the Raw final score at an earlier x position, and its diamond endpoint is within the predeclared acceptable quality loss. The plots alone do not remove training randomness; repeat the three runs with more seeds before making a strong claim.
